# Ridge Encoding Models — All Embedding Modes

Fits ridge/banded regression encoding models predicting ECoG activity from each
embedding mode produced by notebooks 01–04c, for every subject, electrode and
time lag, with an exact circular-shift permutation null for significance.

## Where this fits in the pipeline

Notebook 05 turns each mode's Hebrew/Arabic embeddings into a **residual**
(the part not linearly predictable from English) plus a component budget for
how much of that residual to keep. This notebook consumes those residuals: for
every mode it fits English-alone, Hebrew-alone, Arabic-alone, and
English+residual conditions against the real ECoG recordings, and pairs each
residual condition with a **dimensionality-matched shift control**, the same
residual block, circularly shifted in time, so it carries identical width but
no real word-to-brain correspondence. Notebook 07 compares real vs. shift to
test whether the residual carries genuine neural information, rather than
just added design capacity.

## What this does

For each subject and each condition:
1. Build the design matrix `[English | Foreign]` for that condition's language mode.
2. Reduce it fold-wise: the English block to `K_EN` PCA components, and the
   residual block (fit on training rows only) to the component budget written
   by notebook 05.
3. Fit banded ridge regression against ECoG activity at each time lag,
   cross-validated, and score with Pearson r per fold/electrode/lag.
4. Test significance with an exact circular-shift permutation null.

## Modes processed

| Mode             | Source     | Shape        | Notes                           |
|------------------|------------|--------------|---------------------------------|
| `sliding_window` | NB 04      | (1735, 768)  | XLM-RoBERTa, ±8-word window    |
| `contextual`     | NB 01–03   | (N, 768)     | XLM-RoBERTa, sentence context  |
| `xglm`           | NB 04b     | (1735, 2048) | XGLM-1.7B, full causal context |
| `gemmax2`        | NB 04c     | (1735, 2304) | GemmaX2-28-2B, full causal context |
| `fasttext`       | Eyal       | (1735, 150)  | Static FastText, no context     |

Modes whose input files are not found are skipped with a warning.
This means the notebook always runs to completion regardless of which
notebooks have been run upstream.

## 1. Imports

In [ ]:
import sys
import os
import json
import io
import warnings
from contextlib import redirect_stdout

sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
from tqdm import tqdm
import mne
from mne_bids import BIDSPath

# block_pipeline.py must sit next to static_encoding.py
from static_encoding import process_embeddings, circular_null

print('Imports ready.')

## 2. Configuration

In [ ]:
DATA_DIR        = '../data/processed/'
WORD_LEVEL_PATH = '../data/sentences/translated_podcast_transcript_filtered.csv'
FT_PATH         = '../data/processed/podcast_trilingual_embeddings.csv'
RESULTS_DIR     = '../results/'
ELECTRODES_XLSX = '../data/Interesting Electrodes.xlsx'
BIDS_ROOT       = '../data/ds005574/derivatives/ecogprep'

# Bump this whenever the pipeline changes. Output folders are versioned by it so a
# new run can never silently reuse files written by an older, incompatible pipeline.
RUN_TAG = 'final_sm200'

freq       = 64
tmin, tmax = -2.0, 2.0

# Global PCA is OFF. Reduction is per-block and fold-wise (see feature_blocks).
use_PCA = False
PCA_dim = 150          # retained only for the signature; unused when use_PCA=False
K_EN    = 150          # components kept for the English block in every condition

# Circular-shift null: drop shifts smaller than this many words.
NULL_EXCLUDE = 50

SMOKE_TEST  = False
SMOKE_SUBJ  = ['05', '03']
SMOKE_MODES = ['fasttext']

subjects = ['01', '02', '03', '04', '05', '06', '07', '08', '09']
if SMOKE_TEST:
    subjects = SMOKE_SUBJ

os.makedirs(RESULTS_DIR, exist_ok=True)

MANIFEST_PATH = DATA_DIR + 'block_manifest.json'
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH) as f:
        BLOCK_MANIFEST = json.load(f)
    print(f'Loaded block manifest ({len(BLOCK_MANIFEST)} entries):')
    for k, v in BLOCK_MANIFEST.items():
        flag = '  (capped)' if v.get('capped') else ''
        print(f"  {k:24s} k={v['res_k']:4d}  retains {v['res_var_retained']:.1%}{flag}")
else:
    BLOCK_MANIFEST = {}
    print(f'WARNING: {MANIFEST_PATH} not found — run notebook 05 first.')

print()
print(f'Run tag          : {RUN_TAG}')
print(f'Subjects         : {subjects}')
print(f'Freq / tmin-tmax : {freq}Hz / ({tmin}, {tmax})')
print(f'Global PCA       : {use_PCA}   (per-block, fold-wise reduction instead)')
print(f'English block    : {K_EN} components')
print(f'Null exclusion   : {NULL_EXCLUDE} words')

## 3. Electrode Selection

Applied **before** encoding, from the advisor's pre-specified language-responsive list.
Because selection no longer depends on any model's performance, the circularity concern
that applied to threshold-based selection does not arise here.

Subjects with no matching electrodes are dropped automatically.

In [ ]:
if not os.path.exists(ELECTRODES_XLSX):
    raise FileNotFoundError(
        f'{ELECTRODES_XLSX} not found. This is the advisor\'s pre-specified '
        'language-responsive electrode list (sheet "tagging") and is required '
        'to run electrode selection.'
    )
tagging_df = pd.read_excel(ELECTRODES_XLSX, sheet_name='tagging')

selected_channels = {}   # subj -> list of channel NAMES, in raw order

print(f'Electrode list : {ELECTRODES_XLSX} (sheet: tagging)')
print()
print(f'{"Subj":>6s}  {"In raw":>8s}  {"Listed":>8s}  {"Matched":>9s}  Unmatched')
print('-' * 60)

for subj in subjects:
    fif_path = str(BIDSPath(
        root=BIDS_ROOT, subject=subj, task='podcast', datatype='ieeg',
        description='highgamma', suffix='ieeg', extension='.fif').fpath)

    if not os.path.exists(fif_path):
        print(f'{subj:>6s}  .fif not found')
        continue

    raw_info  = mne.io.read_raw_fif(fif_path, verbose=False).info
    ch_names  = raw_info['ch_names']
    listed    = tagging_df.loc[tagging_df['Podcast Subject'] == f'sub-{subj}',
                               'Electrode'].tolist()

    # keep raw's channel order so saved arrays are unambiguous
    matched   = [ch for ch in ch_names if ch in set(listed)]
    unmatched = [e for e in listed if e not in set(ch_names)]

    if matched:
        selected_channels[subj] = matched

    print(f'{subj:>6s}  {len(ch_names):>8d}  {len(listed):>8d}  {len(matched):>9d}  '
          + (str(unmatched[:6]) + ('...' if len(unmatched) > 6 else '') if unmatched else ''))

subjects = [s for s in subjects if s in selected_channels]

sel_path = os.path.join(RESULTS_DIR, 'selected_electrodes_tagging.json')
with open(sel_path, 'w') as f:
    json.dump(selected_channels, f, indent=2)

total = sum(len(v) for v in selected_channels.values())
print()
print(f'Usable subjects : {subjects}')
print(f'Total electrodes: {total}')
print(f'Saved           : {sel_path}')

## 4. Load All Embeddings

In [ ]:
full_transcript = pd.read_csv(WORD_LEVEL_PATH)
word_df_all     = full_transcript[['start', 'end']].reset_index(drop=True)


def parse_embedding(x):
    if isinstance(x, str):
        x = x.replace('[', '').replace(']', '')
        return np.array(x.split(), dtype=np.float32)
    return np.array(x, dtype=np.float32)


def try_load_csv(path):
    return pd.read_csv(path).values.astype(np.float32) if os.path.exists(path) else None


# MODE_DATA[mode] = dict(E=, H=, A=, wdf=, suffix=)
MODE_DATA = {}

# -- FastText (static) --------------------------------------------------------
if os.path.exists(FT_PATH):
    ft_df = pd.read_csv(FT_PATH)
    MODE_DATA['fasttext'] = dict(
        E=np.vstack(ft_df['en_embedding'].apply(parse_embedding).values),
        H=np.vstack(ft_df['he_embedding'].apply(parse_embedding).values),
        A=np.vstack(ft_df['ar_embedding'].apply(parse_embedding).values),
        wdf=word_df_all, suffix='ft')

# -- Contextual XLM-RoBERTa (sentence level, EN/HE/AR intersection) -----------
ctx_files = ['contextual_shared_indices.csv',
             'en_contextual_aligned_embeddings.csv',
             'he_contextual_aligned_embeddings.csv',
             'ar_contextual_aligned_embeddings.csv',
             'en_contextual_matched_indices.csv',
             'he_contextual_matched_indices.csv',
             'ar_contextual_matched_indices.csv']
if all(os.path.exists(DATA_DIR + f) for f in ctx_files):
    shared = pd.read_csv(DATA_DIR + 'contextual_shared_indices.csv')['original_word_idx'].values
    pos = {}
    for lg in ['en', 'he', 'ar']:
        idx = pd.read_csv(DATA_DIR + f'{lg}_contextual_matched_indices.csv')['original_word_idx'].values
        pos[lg] = {v: i for i, v in enumerate(idx)}
    MODE_DATA['contextual'] = dict(
        E=try_load_csv(DATA_DIR + 'en_contextual_aligned_embeddings.csv')[[pos['en'][i] for i in shared]],
        H=try_load_csv(DATA_DIR + 'he_contextual_aligned_embeddings.csv')[[pos['he'][i] for i in shared]],
        A=try_load_csv(DATA_DIR + 'ar_contextual_aligned_embeddings.csv')[[pos['ar'][i] for i in shared]],
        wdf=full_transcript.iloc[shared][['start', 'end']].reset_index(drop=True),
        suffix='ctx')

# -- Sliding window / XGLM / GemmaX2 -----------------------------------------
for mode, prefix, suffix in [('sliding_window', 'sliding_window', 'sw'),
                             ('xglm',           'xglm',           'xglm'),
                             ('gemmax2',        'gemmax2',        'gemmax2')]:
    files = [DATA_DIR + f'{lg}_{prefix}_embeddings.csv' for lg in ['en', 'he', 'ar']]
    if all(os.path.exists(f) for f in files):
        MODE_DATA[mode] = dict(E=try_load_csv(files[0]), H=try_load_csv(files[1]),
                               A=try_load_csv(files[2]), wdf=word_df_all, suffix=suffix)

if SMOKE_TEST:
    MODE_DATA = {k: v for k, v in MODE_DATA.items() if k in SMOKE_MODES}

print(f'{"Mode":16s}  {"Words":>6s}  {"Dim":>6s}')
print('-' * 34)
for mode, d in MODE_DATA.items():
    assert d['E'].shape == d['H'].shape == d['A'].shape, f'{mode}: shape mismatch'
    assert len(d['wdf']) == len(d['E']), f'{mode}: word_df / embedding length mismatch'
    print(f'{mode:16s}  {len(d["E"]):>6d}  {d["E"].shape[1]:>6d}d')

missing = {'fasttext', 'contextual', 'sliding_window', 'xglm', 'gemmax2'} - set(MODE_DATA)
if missing and not SMOKE_TEST:
    print(f'\nNot found, will be skipped: {sorted(missing)}')

## 5. Define All Conditions

`feature_blocks` describes how the raw design matrix is split and reduced *inside each
training fold*:

- `en` block: PCA to `K_EN` components (passthrough when the block is already <= K_EN,
  which is the case for FastText's native 150d — matching the original pipeline, which
  also left it unreduced).
- `residual` block: fit `W` on the training rows, take `Foreign - English @ W`, then PCA
  to the budget from notebook 05's manifest.

`language_mode` still controls what `process_embeddings` concatenates, so the column
layout the blocks refer to is `[English | Foreign]`.

In [ ]:
def en_block(n_en):
    return {'kind': 'pca', 'cols': (0, n_en),
            'k': None if n_en <= K_EN else K_EN}

def blocks_single(n):
    return [en_block(n)]

def blocks_residual(n_en, n_fo, k_res):
    return [en_block(n_en),
            {'kind': 'residual',
             'en_cols': (0, n_en), 'foreign_cols': (n_en, n_en + n_fo),
             'k': k_res}]


def out_folder(mode):
    return os.path.join(RESULTS_DIR,
                        f'encoding_{mode}_{RUN_TAG}_{freq}Hz_({tmin},{tmax})')


# (name, language_mode, en_mat, he_mat, ar_mat, word_df, out_folder, feature_blocks)
MODES_CONDITIONS = []

# Pairs a residual condition with its dimensionality-matched shift control, so
# notebook 07 knows which difference to test.
RESIDUAL_PAIRS = []          # (mode, real_condition, shift_condition)

for mode, d in MODE_DATA.items():
    E, H, A, wdf, sfx = d['E'], d['H'], d['A'], d['wdf'], d['suffix']
    n, nw = E.shape[1], len(wdf)
    out = out_folder(mode)
    os.makedirs(out, exist_ok=True)

    k_he = BLOCK_MANIFEST.get(f'{mode}_he', {}).get('res_k')
    k_ar = BLOCK_MANIFEST.get(f'{mode}_ar', {}).get('res_k')
    if k_he is None or k_ar is None:
        print(f'{mode}: missing manifest entry — some residual conditions skipped.')

    sh = nw // 2   # cyclic shift for the dimensionality-matched residual null

    MODES_CONDITIONS += [
        (f'en_{sfx}', 'en', E, H, A, wdf, out, blocks_single(n)),
        (f'he_{sfx}', 'he', E, H, A, wdf, out, blocks_single(n)),
        (f'ar_{sfx}', 'ar', E, H, A, wdf, out, blocks_single(n)),
    ]
    if k_he is not None:
        MODES_CONDITIONS += [
            (f'en+he_{sfx}_res',       'en+he', E, H,                 A, wdf, out, blocks_residual(n, n, k_he)),
            (f'en+he_{sfx}_res_shift', 'en+he', E, np.roll(H, sh, 0), A, wdf, out, blocks_residual(n, n, k_he)),
        ]
        RESIDUAL_PAIRS.append((mode, f'en+he_{sfx}_res', f'en+he_{sfx}_res_shift'))
    if k_ar is not None:
        MODES_CONDITIONS += [
            (f'en+ar_{sfx}_res',       'en+ar', E, H, A,                 wdf, out, blocks_residual(n, n, k_ar)),
            (f'en+ar_{sfx}_res_shift', 'en+ar', E, H, np.roll(A, sh, 0), wdf, out, blocks_residual(n, n, k_ar)),
        ]
        RESIDUAL_PAIRS.append((mode, f'en+ar_{sfx}_res', f'en+ar_{sfx}_res_shift'))

# ── FastText restricted to the contextual model's shared word set ────────────
# Table 3 in notebook 07 compares FastText against the contextual model; since
# they don't cover the same words, this condition re-scores FastText on just
# the contextual model's word set, as a word-set control.
shared_idx_path = DATA_DIR + 'contextual_shared_indices.csv'
if 'fasttext' in MODE_DATA and os.path.exists(shared_idx_path):
    shared = pd.read_csv(shared_idx_path)['original_word_idx'].values
    E_ft_s = MODE_DATA['fasttext']['E'][shared]
    H_ft_s = MODE_DATA['fasttext']['H'][shared]
    A_ft_s = MODE_DATA['fasttext']['A'][shared]
    wdf_s  = full_transcript.iloc[shared][['start', 'end']].reset_index(drop=True)
    MODES_CONDITIONS.append(
        ('en_ft_shared', 'en', E_ft_s, H_ft_s, A_ft_s, wdf_s,
         out_folder('fasttext'), blocks_single(E_ft_s.shape[1]))
    )

with open(os.path.join(RESULTS_DIR, f'residual_pairs_{RUN_TAG}.json'), 'w') as f:
    json.dump(RESIDUAL_PAIRS, f, indent=2)

print(f'Conditions : {len(MODES_CONDITIONS)}')
print(f'Subjects   : {len(subjects)}')
print(f'Total runs : {len(subjects) * len(MODES_CONDITIONS)}')
print()
print(f'{"Condition":26s}  {"Mode":8s}  {"Words":>6s}  {"Raw dim":>8s}  Design after reduction')
print('-' * 92)
for name, mode, E, H, A, wdf, folder, blocks in MODES_CONDITIONS:
    width = sum(b['cols'][1] - b['cols'][0] if b['kind'] == 'pca'
                else b['foreign_cols'][1] - b['foreign_cols'][0] for b in blocks)
    kept = sum((b['cols'][1] - b['cols'][0]) if (b['kind'] == 'pca' and
               (b.get('k') is None or b['k'] >= b['cols'][1] - b['cols'][0])) else b['k']
               for b in blocks)
    print(f'{name:26s}  {mode:8s}  {len(wdf):>6d}  {width:>8d}  -> {kept}')

## 6. Encoding Loop

For each subject the ECoG file is loaded, restricted to the pre-specified electrodes,
and every pending condition is run. Two files are written per condition:

- `corrs subj=... .npy` — per-fold correlations, shape `(n_folds, n_electrodes, n_times)`
- `null subj=... .npz`  — observed r, permutation p, and the 95th/99th null percentiles

Already-completed conditions are skipped, so the cell is safe to re-run after an
interruption.

In [ ]:
for subj in subjects:
    fif_path = str(BIDSPath(
        root=BIDS_ROOT, subject=subj, task='podcast', datatype='ieeg',
        description='highgamma', suffix='ieeg', extension='.fif').fpath)

    if not os.path.exists(fif_path):
        print(f'Subject {subj}: .fif not found, skipping.')
        continue

    pending = [c for c in MODES_CONDITIONS
               if not os.path.exists(os.path.join(c[6], f'corrs subj={subj} - {c[0]}.npy'))]
    if not pending:
        print(f'Subject {subj}: all {len(MODES_CONDITIONS)} conditions already saved.')
        continue

    raw      = mne.io.read_raw_fif(fif_path, verbose=False)
    raw      = raw.pick(selected_channels[subj])
    channels = list(raw.info['ch_names'])

    print(f'\nSubject {subj}: {len(channels)} electrodes | '
          f'{len(pending)}/{len(MODES_CONDITIONS)} conditions to run')

    for name, mode, en_mat, he_mat, ar_mat, base_df, outpath, blocks in tqdm(
            pending, desc=f'Subj {subj}'):

        embedding_df = base_df.copy()
        embedding_df['en_embedding'] = list(en_mat.astype(np.float32))
        embedding_df['he_embedding'] = list(he_mat.astype(np.float32))
        embedding_df['ar_embedding'] = list(ar_mat.astype(np.float32))

        buf = io.StringIO()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            with redirect_stdout(buf):
                _, cv_scores, Y_true, Y_pred = process_embeddings(
                    embedding_df=embedding_df, raw=raw,
                    channel_names_regex='', freq=freq,
                    tmin=tmin, tmax=tmax, language_mode=mode,
                    random_noise_mode='over all embeds',
                    use_PCA=use_PCA, PCA_dim=PCA_dim,
                    feature_blocks=blocks,
                )

        # Guard: the saved array must describe exactly the selected electrodes.
        assert cv_scores.shape[1] == len(channels), (
            f'{name}: cv_scores has {cv_scores.shape[1]} electrode rows but '
            f'{len(channels)} channels were selected — stale file or wrong pick.')

        # Guard: the global PCA must not have fired.
        if 'Before PCA' in buf.getvalue():
            raise RuntimeError(
                f'{name}: global PCA ran despite use_PCA=False. Check the '
                'indentation of the PCA block in static_encoding.py.')

        np.save(os.path.join(outpath, f'corrs subj={subj} - {name}.npy'), cv_scores)
        np.save(os.path.join(outpath, f'channels subj={subj}.npy'), np.array(channels))

        # Exact circular-shift null. p_electrode follows Goldstein et al. 2022:
        # observed max-across-lags scored against a null of max-across-lags-and-
        # electrodes, so the positive bias of the max operator cancels.
        nullres = circular_null(Y_true, Y_pred, cv_scores.shape[1:],
                                exclude=NULL_EXCLUDE)
        np.savez_compressed(
            os.path.join(outpath, f'null subj={subj} - {name}.npz'),
            observed    = nullres['observed'],
            p_lagwise   = nullres['p_lagwise'],
            p_electrode = nullres['p_electrode'],
            null_max    = nullres['null_max'],
            q95         = nullres['q95'],
            q99         = nullres['q99'],
            n_words     = len(Y_true),
            exclude     = NULL_EXCLUDE)

print('\nEncoding loop complete.')

## 7. Summary Tables

**Table 1** — encoding performance per condition, with the circular-shift permutation
p-value. That p answers *"does this condition predict brain activity above chance?"*

**Table 2** — residual contrasts. The shift control still contains the real English
block, so it predicts above chance by construction; the permutation p is therefore the
wrong test for the residual hypothesis. The correct statistic is the paired difference
`r(residual) - r(shift control)`, which holds dimensionality, scale and covariance
constant and varies only the word-to-brain correspondence of the residual block.

Notebook 07 applies FDR-corrected paired t-tests across subjects to both tables.


In [ ]:
from statsmodels.stats.multitest import fdrcorrection

FDR_Q = 0.01     # Goldstein et al. 2022 threshold
WINDOW = (0.0, 0.5)     # pre-specified, post word onset

def stats_per_subject(folder, cond_name):
    win_r, gpeak, peak_lag, frac_sig = {}, {}, {}, {}
    for subj in subjects:
        cf = os.path.join(folder, f'corrs subj={subj} - {cond_name}.npy')
        if not os.path.exists(cf):
            continue
        cv   = np.load(cf).mean(0)
        lags = np.linspace(tmin, tmax, cv.shape[-1])
        gavg = cv.mean(0)
        m    = (lags >= WINDOW[0]) & (lags <= WINDOW[1])
        win_r[subj]    = float(gavg[m].mean())      # primary — no max anywhere
        gpeak[subj]    = float(gavg.max())          # secondary
        peak_lag[subj] = float(lags[gavg.argmax()])
        nf = os.path.join(folder, f'null subj={subj} - {cond_name}.npz')
        if os.path.exists(nf):
            z = np.load(nf)
            rej, _ = fdrcorrection(np.asarray(z['p_electrode']).ravel(), alpha=FDR_Q)
            frac_sig[subj] = float(rej.mean())
    return win_r, gpeak, peak_lag, frac_sig


def mean_or_nan(d):
    return np.mean(list(d.values())) if d else float('nan')


# ── Table 1: encoding performance and significance vs chance ─────────────────
rows = []
for mode in MODE_DATA:
    folder = out_folder(mode)
    for cond in [c[0] for c in MODES_CONDITIONS if c[6] == folder]:
        w, g, l, s = stats_per_subject(folder, cond)
        rows.append({
            'mode'         : mode,
            'condition'    : cond,
            'win_r'        : mean_or_nan(w),
            'gavg_peak_r'  : mean_or_nan(g),
            'peak_lag_s'   : mean_or_nan(l),
            'frac_sig_elec': mean_or_nan(s),
            'n_subj'       : len(w),
        })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, f'encoding_summary_{RUN_TAG}.csv'), index=False)

print('=' * 80)
print('TABLE 1 — encoding performance')
print(f'  win_r         : mean r over {WINDOW[0]}-{WINDOW[1]}s post-onset (primary — no max anywhere)')
print('  gavg_peak_r   : peak of the electrode-averaged lag curve (secondary)')
print('  peak_lag_s    : lag of that peak, in seconds (expect ~0.15-0.40 s)')
print(f'  frac_sig_elec : fraction of electrodes significant, FDR q<{FDR_Q}')
print('=' * 80)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))


# ── Table 2: does the residual add anything beyond English? ──────────────────
# The shift control still contains the real English block, so it predicts above
# chance by construction — the permutation p is the wrong test here. The correct
# statistic is the paired difference against the dimensionality-matched control.
pair_rows = []
for mode, real_cond, shift_cond in RESIDUAL_PAIRS:
    folder  = out_folder(mode)
    en_cond = f'en_{MODE_DATA[mode]["suffix"]}'
    w_real,  _, _, _ = stats_per_subject(folder, real_cond)
    w_shift, _, _, _ = stats_per_subject(folder, shift_cond)
    w_en,    _, _, _ = stats_per_subject(folder, en_cond)
    shared = sorted(set(w_real) & set(w_shift) & set(w_en))
    if not shared:
        continue
    pair_rows.append({
        'mode'             : mode,
        'condition'        : real_cond,
        'r_real'           : np.mean([w_real[s]  for s in shared]),
        'r_shift_control'  : np.mean([w_shift[s] for s in shared]),
        'r_english_only'   : np.mean([w_en[s]    for s in shared]),
        'delta_vs_shift'   : np.mean([w_real[s] - w_shift[s] for s in shared]),
        'delta_vs_english' : np.mean([w_real[s] - w_en[s]    for s in shared]),
        'n_subj'           : len(shared),
    })

pairs_df = pd.DataFrame(pair_rows)
pairs_df.to_csv(os.path.join(RESULTS_DIR, f'residual_contrasts_{RUN_TAG}.csv'), index=False)

print()
print('=' * 80)
print('TABLE 2 — residual contrasts')
print('  r_real, r_shift_control, r_english_only : win_r (primary metric from Table 1)')
print('  delta_vs_shift   : residual beyond a dimensionality-matched null  (the test)')
print('  delta_vs_english : residual beyond English alone  (confounded by width)')
print('  FDR-corrected paired t-tests across subjects are applied in notebook 07.')
print('=' * 80)
print(pairs_df.to_string(index=False, float_format='{:.4f}'.format) if len(pairs_df)
      else '(no complete residual pairs yet)')

print()
print(f'Saved: {os.path.abspath(RESULTS_DIR)}')
print('Proceed to notebook 07 for figures and FDR-corrected statistics.')